# LLM Evaluator Variability Analysis

This notebook analyzes the results from the variability study experiment.
It reads the CSV files generated by the experiment and performs:
- Variability metric calculations
- Visualizations
- Statistical tests

## Experimental Conditions
- **Sequential (no batch):** 1 worker per GPU - no batching
- **Parallel (with batch):** 5 workers per GPU - with batching

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 10

In [ ]:
# Configuration
INPUT_DIR = './replicas/replicas_temp_NO_0/replicas_seed'
OUTPUT_DIR = './analysis_results'
Path(OUTPUT_DIR).mkdir(exist_ok=True)

MODEL_LIST = ["llama3.1:latest", "phi4:latest", "qwen2.5:7b", "deepseek-r1:7b", "llama3.3:latest", "deepseek-r1:70b"]
N_REPLICAS = 5

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Models to analyze: {MODEL_LIST}")
print(f"Number of replicas: {N_REPLICAS}")

## Load Data from CSV Files

In [ ]:
def load_all_results(input_dir, model_list, modes=['sequential', 'parallel']):
    """
    Load all experiment results from CSV files.
    
    Args:
        input_dir: Directory containing CSV files
        model_list: List of model names
        modes: List of execution modes ['sequential', 'parallel']
    
    Returns:
        Dictionary with results organized by model and mode
    """
    results = {}
    
    for model_name in model_list:
        results[model_name] = {}
        model_safe = model_name.replace(':', '_')
        
        for mode in modes:
            # Try to load the combined file for this model and mode
            file_path = Path(input_dir) / f"{mode}_{model_safe}_all_replicas.csv"
            
            if file_path.exists():
                df = pd.read_csv(file_path)
                results[model_name][mode] = df
                print(f"✓ Loaded {mode} results for {model_name}: {len(df)} evaluations")
            else:
                print(f"✗ File not found: {file_path}")
                results[model_name][mode] = None
    
    return results

# Load all results
all_results = load_all_results(INPUT_DIR, MODEL_LIST)

print(f"\n{'='*80}")
print("Data loading complete")
print(f"{'='*80}")

## Calculate Variability Metrics

For each answer across replicas, calculate:
- Standard deviation (std)
- Range (max - min)
- Coefficient of variation (CV)

In [ ]:
def calculate_variability_metrics(df, mode, model):
    """
    Calculate variability metrics for a given mode and model.
    
    Args:
        df: DataFrame with evaluation results
        mode: Execution mode ('sequential' or 'parallel')
        model: Model name
    
    Returns:
        DataFrame with variability metrics per answer
    """
    # Filter successful evaluations
    df_success = df[df['success'] == True].copy()
    
    if len(df_success) == 0:
        print(f"Warning: No successful evaluations found for {model} ({mode})")
        return pd.DataFrame()
    
    # Calculate variability per answer
    variability_data = []
    
    for answer_id in df_success['answer_id'].unique():
        answer_df = df_success[df_success['answer_id'] == answer_id]
        
        if len(answer_df) < 2:
            continue
        
        # Calculate standard deviation
        accuracy_std = answer_df['accuracy_score'].std()
        clarity_std = answer_df['clarity_score'].std()
        completeness_std = answer_df['completeness_score'].std()
        
        # Calculate range
        accuracy_range = answer_df['accuracy_score'].max() - answer_df['accuracy_score'].min()
        clarity_range = answer_df['clarity_score'].max() - answer_df['clarity_score'].min()
        completeness_range = answer_df['completeness_score'].max() - answer_df['completeness_score'].min()
        
        # Calculate coefficient of variation (CV)
        accuracy_mean = answer_df['accuracy_score'].mean()
        clarity_mean = answer_df['clarity_score'].mean()
        completeness_mean = answer_df['completeness_score'].mean()
        
        accuracy_cv = (accuracy_std / accuracy_mean) * 100 if accuracy_mean > 0 else 0
        clarity_cv = (clarity_std / clarity_mean) * 100 if clarity_mean > 0 else 0
        completeness_cv = (completeness_std / completeness_mean) * 100 if completeness_mean > 0 else 0
        
        variability_data.append({
            'answer_id': answer_id,
            'mode': mode,
            'model': model,
            'n_replicas': len(answer_df),
            'accuracy_std': accuracy_std,
            'clarity_std': clarity_std,
            'completeness_std': completeness_std,
            'accuracy_range': accuracy_range,
            'clarity_range': clarity_range,
            'completeness_range': completeness_range,
            'accuracy_cv': accuracy_cv,
            'clarity_cv': clarity_cv,
            'completeness_cv': completeness_cv,
            'accuracy_mean': accuracy_mean,
            'clarity_mean': clarity_mean,
            'completeness_mean': completeness_mean
        })
    
    return pd.DataFrame(variability_data)

# Calculate variability for all models and modes
all_variability = []

for model_name in MODEL_LIST:
    for mode in ['sequential', 'parallel']:
        df = all_results[model_name].get(mode)
        if df is not None and len(df) > 0:
            var_df = calculate_variability_metrics(df, mode, model_name)
            if len(var_df) > 0:
                all_variability.append(var_df)
                print(f"✓ Calculated variability for {model_name} ({mode}): {len(var_df)} answers")

# Combine all variability data
df_variability = pd.concat(all_variability, ignore_index=True)

# Save variability metrics
output_file = f"{OUTPUT_DIR}/variability_metrics_summary.csv"
df_variability.to_csv(output_file, index=False)
print(f"\n✓ Variability metrics saved: {output_file}")
print(f"Total answers analyzed: {len(df_variability)}")

# Display summary
print(f"\n{'='*80}")
print("Variability Metrics Summary")
print(f"{'='*80}")
print(df_variability.groupby(['model', 'mode'])[['accuracy_std', 'clarity_std', 'completeness_std']].agg(['mean', 'median']).round(3))

## Summary Statistics

In [ ]:
# Summary statistics
print("\n" + "="*80)
print("VARIABILITY SUMMARY STATISTICS")
print("="*80)

summary_stats = df_variability.groupby(['model', 'mode']).agg({
    'accuracy_std': ['mean', 'median', 'std'],
    'clarity_std': ['mean', 'median', 'std'],
    'completeness_std': ['mean', 'median', 'std'],
    'accuracy_range': ['mean', 'median', 'max'],
    'clarity_range': ['mean', 'median', 'max'],
    'completeness_range': ['mean', 'median', 'max']
}).round(3)

print(summary_stats)

# Save summary
summary_file = f"{OUTPUT_DIR}/variability_summary_stats.csv"
summary_stats.to_csv(summary_file)
print(f"\n✓ Summary statistics saved: {summary_file}")

## Visualization 1: Standard Deviation Comparison

In [ ]:
# Plot 1: Standard deviation comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['accuracy_std', 'clarity_std', 'completeness_std']
titles = ['Accuracy Variability (Std Dev)', 'Clarity Variability (Std Dev)', 'Completeness Variability (Std Dev)']

for ax, metric, title in zip(axes, metrics, titles):
    sns.boxplot(data=df_variability, x='model', y=metric, hue='mode', ax=ax, palette='Set2')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Model', fontsize=10)
    ax.set_ylabel('Standard Deviation', fontsize=10)
    ax.legend(title='Mode', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
output_file = f"{OUTPUT_DIR}/figure_variability_std_comparison.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

## Visualization 2: Score Range Comparison

In [ ]:
# Plot 2: Score range comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['accuracy_range', 'clarity_range', 'completeness_range']
titles = ['Accuracy Range', 'Clarity Range', 'Completeness Range']

for ax, metric, title in zip(axes, metrics, titles):
    sns.violinplot(data=df_variability, x='model', y=metric, hue='mode', ax=ax, palette='Set1', split=True)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Model', fontsize=10)
    ax.set_ylabel('Score Range (Max - Min)', fontsize=10)
    ax.legend(title='Mode', fontsize=9)
    ax.set_ylim(0, 7)
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
output_file = f"{OUTPUT_DIR}/figure_variability_range_comparison.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

## Visualization 3: Coefficient of Variation

In [ ]:
# Plot 3: Coefficient of variation with p-values table
from scipy.stats import mannwhitneyu

fig = plt.figure(figsize=(18, 7))
gs = fig.add_gridspec(2, 3, height_ratios=[4, 1], hspace=0.3)

metrics = ['accuracy_cv', 'clarity_cv', 'completeness_cv']
titles = ['Accuracy CV (%)', 'Clarity CV (%)', 'Completeness CV (%)']

# Store p-values for table
pvalue_data = {metric: [] for metric in metrics}

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = fig.add_subplot(gs[0, idx])
    
    # Create grouped bar plot
    data_pivot = df_variability.groupby(['model', 'mode'])[metric].mean().unstack()
    data_pivot.plot(kind='bar', ax=ax, color=['#4472C4', '#ED7D31'], alpha=0.8)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('Coefficient of Variation (%)', fontsize=10)
    ax.legend(title='Mode', fontsize=9, loc='upper right')
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels([])  # Remove x labels for cleaner look
    
    # Calculate p-values
    for model in MODEL_LIST:
        seq_data = df_variability[(df_variability['model'] == model) & 
                                   (df_variability['mode'] == 'sequential')][metric].dropna()
        par_data = df_variability[(df_variability['model'] == model) & 
                                   (df_variability['mode'] == 'parallel')][metric].dropna()
        
        if len(seq_data) > 0 and len(par_data) > 0:
            statistic, p_value = mannwhitneyu(seq_data, par_data, alternative='two-sided')
            sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else 'ns'
            pvalue_data[metric].append(f'{sig}\n{p_value:.3f}')
        else:
            pvalue_data[metric].append('N/A')
    
    # Add p-value table below
    ax_table = fig.add_subplot(gs[1, idx])
    ax_table.axis('tight')
    ax_table.axis('off')
    
    table_data = [[model, pvalue_data[metric][i]] for i, model in enumerate(MODEL_LIST)]
    table = ax_table.table(cellText=table_data,
                          colLabels=['Model', 'p-value'],
                          cellLoc='center',
                          loc='center',
                          colWidths=[0.6, 0.4])
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.5)
    
    # Style header
    for i in range(2):
        table[(0, i)].set_facecolor('#D3D3D3')
        table[(0, i)].set_text_props(weight='bold')

plt.tight_layout()
output_file = f"{OUTPUT_DIR}/figure_variability_cv_comparison_with_pvalues.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

print("\n*** p<0.001 | ** p<0.01 | * p<0.05 | ns p≥0.05")

## Visualization 4: Heatmap of Mean Variability

In [ ]:
# Plot 4: Heatmap of mean variability
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
metrics = ['accuracy_std', 'clarity_std', 'completeness_std']
titles = ['Accuracy Std Dev', 'Clarity Std Dev', 'Completeness Std Dev']

for ax, metric, title in zip(axes, metrics, titles):
    pivot_data = df_variability.pivot_table(values=metric, index='model', columns='mode', aggfunc='mean')
    sns.heatmap(pivot_data, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Mean Std Dev'})
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Execution Mode', fontsize=10)
    ax.set_ylabel('Model', fontsize=10)

plt.tight_layout()
output_file = f"{OUTPUT_DIR}/figure_variability_heatmap.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

## Statistical Tests

Compare variability between Sequential (no batch) and Parallel (with batch) modes using:
- Mann-Whitney U test (non-parametric)
- Cohen's d effect size

In [ ]:
print("\n" + "="*80)
print("STATISTICAL TESTS: Sequential (no batch) vs. Parallel (with batch)")
print("="*80)

test_results = []

for model in MODEL_LIST:
    print(f"\n--- {model} ---")
    
    seq_data = df_variability[(df_variability['model'] == model) & (df_variability['mode'] == 'sequential')]
    par_data = df_variability[(df_variability['model'] == model) & (df_variability['mode'] == 'parallel')]
    
    if len(seq_data) == 0 or len(par_data) == 0:
        print(f"  Skipping {model}: insufficient data")
        continue
    
    for metric in ['accuracy_std', 'clarity_std', 'completeness_std']:
        metric_name = metric.replace('_std', '').capitalize()
        
        seq_values = seq_data[metric].dropna()
        par_values = par_data[metric].dropna()
        
        if len(seq_values) > 0 and len(par_values) > 0:
            # Mann-Whitney U test
            statistic, p_value = mannwhitneyu(seq_values, par_values, alternative='two-sided')
            
            # Effect size (Cohen's d)
            pooled_std = np.sqrt((seq_values.std()**2 + par_values.std()**2) / 2)
            cohens_d = (seq_values.mean() - par_values.mean()) / pooled_std if pooled_std > 0 else 0
            
            print(f"  {metric_name}:")
            print(f"    Sequential mean: {seq_values.mean():.3f}, Parallel mean: {par_values.mean():.3f}")
            print(f"    Mann-Whitney U: {statistic:.2f}, p-value: {p_value:.4f}")
            print(f"    Cohen's d: {cohens_d:.3f}")
            
            test_results.append({
                'model': model,
                'metric': metric_name,
                'sequential_mean': seq_values.mean(),
                'sequential_std': seq_values.std(),
                'parallel_mean': par_values.mean(),
                'parallel_std': par_values.std(),
                'mann_whitney_u': statistic,
                'p_value': p_value,
                'cohens_d': cohens_d,
                'significant': 'Yes' if p_value < 0.05 else 'No'
            })

# Save test results
df_tests = pd.DataFrame(test_results)
test_file = f"{OUTPUT_DIR}/statistical_tests_results.csv"
df_tests.to_csv(test_file, index=False)
print(f"\n✓ Statistical test results saved: {test_file}")

In [ ]:
# Violin plots with Mann-Whitney p-values
from scipy.stats import mannwhitneyu
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
metrics = ['accuracy_std', 'clarity_std', 'completeness_std']
titles = ['Accuracy Std Dev', 'Clarity Std Dev', 'Completeness Std Dev']

for ax, metric, title in zip(axes, metrics, titles):
    # Create violin plot
    sns.violinplot(
        data=df_variability, 
        x='model', 
        y=metric, 
        hue='mode', 
        ax=ax, 
        palette={'sequential': 'lightblue', 'parallel': 'lightcoral'},
        split=True,
        inner='quartile'
    )
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Model', fontsize=11)
    ax.set_ylabel('Standard Deviation', fontsize=11)
    ax.legend(title='Mode', fontsize=10, loc='upper right')
    ax.grid(axis='y', alpha=0.3)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    # Calculate and display p-values for each model
    y_max = df_variability[metric].max()
    y_offset = y_max * 0.05
    
    for i, model in enumerate(MODEL_LIST):
        seq_data = df_variability[(df_variability['model'] == model) & 
                                   (df_variability['mode'] == 'sequential')][metric].dropna()
        par_data = df_variability[(df_variability['model'] == model) & 
                                   (df_variability['mode'] == 'parallel')][metric].dropna()
        
        if len(seq_data) > 0 and len(par_data) > 0:
            statistic, p_value = mannwhitneyu(seq_data, par_data, alternative='two-sided')
            
            # Determine significance level
            if p_value < 0.001:
                sig_marker = '***'
            elif p_value < 0.01:
                sig_marker = '**'
            elif p_value < 0.05:
                sig_marker = '*'
            else:
                sig_marker = 'ns'
            
            # Add p-value text above each model
            ax.text(i, y_max + y_offset, f'p={p_value:.3f}\n{sig_marker}', 
                   ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
output_file = f"{OUTPUT_DIR}/figure_violin_comparison_with_pvalues.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved: {output_file}")
plt.show()

# Print significance legend
print("\n" + "="*80)
print("SIGNIFICANCE LEVELS")
print("="*80)
print("  *** : p < 0.001 (highly significant)")
print("  **  : p < 0.01  (very significant)")
print("  *   : p < 0.05  (significant)")
print("  ns  : p ≥ 0.05  (not significant)")
print("="*80)

## Summary Report

In [ ]:
# Box plots with Mann-Whitney p-values using Plotly (paper style - improved)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import mannwhitneyu

# Create subplots with less spacing
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Accuracy Std Dev', 'Clarity Std Dev', 'Completeness Std Dev'),
    horizontal_spacing=0.06
)

metrics = ['accuracy_std', 'clarity_std', 'completeness_std']
colors = {'sequential': '#4472C4', 'parallel': '#ED7D31'}

for idx, metric in enumerate(metrics, start=1):
    # Add box traces for each model and mode
    for mode in ['sequential', 'parallel']:
        for model_idx, model in enumerate(MODEL_LIST):
            data = df_variability[(df_variability['model'] == model) & 
                                 (df_variability['mode'] == mode)][metric].dropna()
            
            # Calculate position offset for side-by-side boxes
            offset = -0.2 if mode == 'sequential' else 0.2
            
            fig.add_trace(
                go.Box(
                    y=data,
                    x=[model_idx + offset] * len(data),
                    name=mode,
                    legendgroup=mode,
                    marker_color=colors[mode],
                    showlegend=(idx == 1 and model_idx == 0),  # Only show legend once
                    boxmean='sd',  # Show mean and std dev
                    line=dict(width=2),
                    marker=dict(size=5, opacity=0.6),
                    width=0.3
                ),
                row=1, col=idx
            )
    
    # Calculate and add p-values as annotations
    for model_idx, model in enumerate(MODEL_LIST):
        seq_data = df_variability[(df_variability['model'] == model) & 
                                   (df_variability['mode'] == 'sequential')][metric].dropna()
        par_data = df_variability[(df_variability['model'] == model) & 
                                   (df_variability['mode'] == 'parallel')][metric].dropna()
        
        if len(seq_data) > 0 and len(par_data) > 0:
            statistic, p_value = mannwhitneyu(seq_data, par_data, alternative='two-sided')
            
            # Determine significance marker
            if p_value < 0.001:
                sig_marker = '***'
            elif p_value < 0.01:
                sig_marker = '**'
            elif p_value < 0.05:
                sig_marker = '*'
            else:
                sig_marker = 'ns'
            
            # Get y position for annotation
            y_max = max(seq_data.max(), par_data.max()) if len(seq_data) > 0 and len(par_data) > 0 else df_variability[metric].max()
            
            # Add annotation with box
            fig.add_annotation(
                x=model_idx,
                y=y_max * 1.15,
                xref=f'x{idx}',
                yref=f'y{idx}',
                text=f'<b>{sig_marker}</b><br>p={p_value:.3f}',
                showarrow=False,
                font=dict(size=11, color='black', family='Arial, sans-serif'),
                align='center',
                bgcolor='rgba(255, 255, 255, 0.9)',
                bordercolor='rgba(128, 128, 128, 0.5)',
                borderwidth=1,
                borderpad=4
            )
    
    # Update x-axis for this subplot
    fig.update_xaxes(
        tickmode='array',
        tickvals=list(range(len(MODEL_LIST))),
        ticktext=MODEL_LIST,
        title_text='Model',
        title_font=dict(size=14),
        showgrid=False,
        showline=True,
        linewidth=2,
        linecolor='black',
        mirror=True,
        tickangle=45,
        tickfont=dict(size=12, family='Arial, sans-serif'),
        row=1, col=idx
    )
    
    # Update y-axis for this subplot
    fig.update_yaxes(
        title_text='Standard Deviation' if idx == 1 else '',
        title_font=dict(size=14),
        showgrid=True,
        gridwidth=0.5,
        gridcolor='rgba(211, 211, 211, 0.5)',
        showline=True,
        linewidth=2,
        linecolor='black',
        mirror=True,
        tickfont=dict(size=12, family='Arial, sans-serif'),
        row=1, col=idx
    )

# Update layout for paper style
fig.update_layout(
    height=550,
    width=1500,
    font=dict(family="Arial, sans-serif", size=13, color="black"),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        title=dict(text='<b>Mode</b>', font=dict(size=14)),
        orientation="h",
        yanchor="top",
        y=-0.15,
        xanchor="center",
        x=0.5,
        font=dict(size=13),
        bgcolor='rgba(255, 255, 255, 0.9)',
        bordercolor='black',
        borderwidth=1.5
    ),
    showlegend=True,
    boxmode='group',
    margin=dict(t=100, b=100)
)

# Update subplot titles with larger font and better positioning
for i, annotation in enumerate(fig['layout']['annotations'][:3]):
    annotation['font'] = dict(size=16, color='black', family='Arial, sans-serif')
    annotation['y'] = 1.05

# Save figures
output_file_html = f"{OUTPUT_DIR}/figure_boxplot_comparison_with_pvalues_plotly.html"
fig.write_html(output_file_html)
print(f"✓ Saved interactive: {output_file_html}")

try:
    output_file_png = f"{OUTPUT_DIR}/figure_boxplot_comparison_with_pvalues_plotly.png"
    fig.write_image(output_file_png, width=1500, height=550, scale=2)
    print(f"✓ Saved static: {output_file_png}")
except Exception as e:
    print(f"⚠ Could not save PNG (install kaleido: pip install kaleido): {e}")

fig.show()

# Print significance legend
print("\n" + "="*80)
print("SIGNIFICANCE LEVELS")
print("="*80)
print("  *** : p < 0.001 (highly significant)")
print("  **  : p < 0.01  (very significant)")
print("  *   : p < 0.05  (significant)")
print("  ns  : p ≥ 0.05  (not significant)")
print("="*80)

In [ ]:
from datetime import datetime

print("\n" + "="*80)
print("ANALYSIS SUMMARY REPORT")
print("="*80)

print(f"\nAnalysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nModels analyzed: {', '.join(MODEL_LIST)}")
print(f"Number of replicas per condition: {N_REPLICAS}")
print(f"Total answers analyzed: {df_variability['answer_id'].nunique()}")
print(f"Total evaluations: {len(df_variability) * N_REPLICAS}")

print("\n--- Key Findings ---")

# Overall variability
print("\n1. Overall Variability (Mean Standard Deviation):")
overall_var = df_variability.groupby('mode')[['accuracy_std', 'clarity_std', 'completeness_std']].mean()
print(overall_var.round(3))

# Most/least variable model
print("\n2. Model Variability Ranking:")
model_var = df_variability.groupby('model')[['accuracy_std', 'clarity_std', 'completeness_std']].mean()
model_var['avg_variability'] = model_var.mean(axis=1)
model_var_sorted = model_var.sort_values('avg_variability')
print(f"   Most stable: {model_var_sorted.index[0]} (avg std: {model_var_sorted['avg_variability'].iloc[0]:.3f})")
print(f"   Least stable: {model_var_sorted.index[-1]} (avg std: {model_var_sorted['avg_variability'].iloc[-1]:.3f})")

# Mode effect
print("\n3. Batching Effect (Sequential vs. Parallel):")
sig_diffs = df_tests[df_tests['significant'] == 'Yes']
print(f"   Significant differences found: {len(sig_diffs)}/{len(df_tests)}")

if len(sig_diffs) > 0:
    print("   Details:")
    for _, row in sig_diffs.iterrows():
        direction = "higher" if row['sequential_mean'] > row['parallel_mean'] else "lower"
        print(f"     {row['model']} - {row['metric']}: No-batch variability {direction} than batch")
        print(f"       (p={row['p_value']:.4f}, d={row['cohens_d']:.3f})")

print(f"\n{'='*80}")
print(f"All results saved in: {OUTPUT_DIR}")
print(f"{'='*80}")